# T09 — ARMA Model — Book: CH06

**Methodology**: Marco Peixeiro, *Time Series Forecasting in Python*, Chapter 6.

### Book-mandated steps:
1. ADF stationarity test → confirm d=0
2. ACF + PACF plots
3. `optimize_ARMA` → select by lowest AIC (SARIMAX)
4. Fit best model → Ljung-Box residuals
5. `rolling_forecast_engine` → walk-forward validation
6. Full test evaluation

In [ ]:
import sys, os
from pathlib import Path
from functools import partial
from itertools import product

ROOT = Path(os.getcwd()).resolve().parents[1]
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Book imports — exactly as CH06 uses them
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.stattools import adfuller
from statsmodels.stats.diagnostic import acorr_ljungbox

from src.models.classical import (
    build_pca_health_index, compute_failure_threshold,
    run_stationarity_report, plot_acf_pacf, smooth_series,
    optimize_ARMA, check_residuals,
    rolling_forecast_engine, predict_rul_arma,
    predict_dataset, RUL_CAP,
)
from src.evaluation.metrics import evaluate, evaluate_per_subset

PROC_DIR    = ROOT / "data" / "processed"
SENSOR_COLS = [f"s{i}" for i in [2,3,4,7,8,9,11,12,13,14,15,17,20,21]]

## 1. Load data + build health_index

In [ ]:
train = pd.read_csv(PROC_DIR / "train_features.csv")
test  = pd.read_csv(PROC_DIR / "test_features.csv")

train, test = build_pca_health_index(train, test, SENSOR_COLS)

# Per-dataset_id thresholds (not global — fixes audit issue)
THRESHOLD_MAP = compute_failure_threshold(train, end_of_life_rul=5, quantile=0.5)
print("Thresholds per dataset:", THRESHOLD_MAP)

## 2. Stationarity check — ADF (CH06)

ARMA requires stationary series (d=0). ADF on health_index confirms this.

In [ ]:
# Stratified ADF across all 4 subsets
stationarity_df = run_stationarity_report(train, n_engines_per_subset=5)
# ARMA requires d=0 — confirm here

## 3. ACF and PACF plots (CH06)

ACF tail-off + PACF tail-off = ARMA(p,q) signature.

In [ ]:
eid  = train[train.dataset_id == 1].engine_id.iloc[0]
raw  = train[train.engine_id == eid].sort_values("cycle").health_index.values
smth      = smooth_series(raw, window=5)
smth_diff = np.diff(smth, n=1)       
plot_acf_pacf(smth_diff, lags=20, title="health_index DIFFERENCED — engine {eid}")
print("ACF tail-off + PACF tail-off => ARMA(p,q) model")

## 4. `optimize_ARMA` — select (p,q) by AIC (CH06 core step)

Book sorts all (p,q) combos by AIC ascending. Lowest AIC wins.

In [ ]:
# (p, q) grid: book uses tqdm in CH06 for all combos sorted by AIC
order_list = list(product(range(1, 5), range(1, 5)))   # (1,1) to (4,4)

arma_aic_df = optimize_ARMA(smth, order_list)
print(arma_aic_df.head(10).to_string(index=False))

best_pq = arma_aic_df.iloc[0]["(p,q)"]
BEST_P, BEST_Q = int(best_pq[0]), int(best_pq[1])
print(f"Best ARMA order: ({BEST_P},{BEST_Q}) — lowest AIC")

## 5. Fit best ARMA + Ljung-Box (CH06 requirement)

In [ ]:
model_fit = SARIMAX(smth, order=(BEST_P, 0, BEST_Q), simple_differencing=False).fit(disp=False)
print(model_fit.summary())

# Ljung-Box — CH06 mandated residual check
lb_result = check_residuals(model_fit.resid, model_name=f"ARMA({BEST_P},{BEST_Q})")

## 6. Rolling forecast — walk-forward validation (CH06 pattern)

In [ ]:
TRAIN_LEN = int(len(smth) * 0.7)
WINDOW    = 1   # book uses window=2 in CH06; 1 gives most accurate walk-forward

pred_arma = rolling_forecast_engine(
    series    = smth,
    train_len = TRAIN_LEN,
    order     = (BEST_P, 0, BEST_Q),
    window    = WINDOW,
)
actual_val = smth[TRAIN_LEN: TRAIN_LEN + len(pred_arma)]

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(smth, color="steelblue", label="observed health_index")
ax.plot(range(TRAIN_LEN, TRAIN_LEN + len(pred_arma)), pred_arma,
        color="darkorange", ls="--", label=f"ARMA({BEST_P},{BEST_Q}) rolling forecast")
ax.axvline(TRAIN_LEN, color="gray", ls=":", label="train/val split")
ax.axhline(THRESHOLD_MAP.get(1, 0), color="red", ls="--", label="failure threshold")
ax.set_xlabel("cycle"); ax.set_ylabel("health_index")
ax.set_title(f"Rolling forecast — ARMA({BEST_P},{BEST_Q}) — engine {eid}")
ax.legend(); plt.tight_layout(); plt.show()

rmse_roll = float(np.sqrt(np.mean((actual_val - pred_arma)**2)))
print(f"Rolling forecast RMSE on health_index: {rmse_roll:.4f}")

## 7. Full test-set evaluation

In [ ]:
predict_fn = partial(predict_rul_arma, p=BEST_P, q=BEST_Q)
y_true, y_pred, dids = predict_dataset(test, predict_fn, THRESHOLD_MAP)

print(f"=== ARMA({BEST_P},{BEST_Q}) Test Results ===")
test_metrics = evaluate_per_subset(y_true, y_pred, dids, model_name=f"ARMA({BEST_P},{BEST_Q})")
display(test_metrics)